# Image Fundamentals for Computer Vision

**Images are just multidimensional arrays of numbers - and you already know how to work with those!**

Everything you learned about NumPy arrays applies directly to image processing.

## Why Understanding Image Representation Matters:

| Concept | Why It Matters for ML |
|---------|----------------------|
| Tensor shape (C, H, W) | Input dimensions to CNN layers |

| Data types (uint8 vs float32) | Numerical stability in training |**Always check your shapes!** Shape mismatches are the #1 source of CV bugs.

| Normalization | Faster convergence, better gradients |

| Augmentation | Regularization, prevents overfitting || TensorFlow | (H, W, C) or (B, H, W, C) | Channels last by default |

| Batching | Efficient GPU utilization || NumPy/PIL | (H, W, C) or (H, W, 3) | Display convention |

| **PyTorch** | (C, H, W) or (B, C, H, W) | Convolution expects channels first |

## The Computer Vision Pipeline:|-----------|-------|-----|

| Framework | Shape | Why |

```

Image Files → Load → Transform → Batch → Model → Predictions## Key PyTorch Convention:

   (JPEG)    (tensor)  (augment)  (GPU)   (CNN)   (classes)

```- Build efficient data pipelines with DataLoader

- Use transforms.v2 for modern best practices

## Learning Objectives- Apply transforms for preprocessing and augmentation

- Load and display images using torchvision
- Understand image representation as tensors (C, H, W)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# torchvision imports
import torchvision
from torchvision import datasets
from torchvision.transforms import v2  # Modern transforms API
from torchvision.io import decode_image, encode_jpeg
from torch.utils.data import DataLoader

plt.style.use("seaborn-v0_8-whitegrid")
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"torchvision version: {torchvision.__version__}")

## 1. Image Representation

**Images in PyTorch are 3D tensors with shape `(C, H, W)`:**

| Dimension | Meaning | Example |
|-----------|---------|--------|
| **C** | Channels | 1 (grayscale), 3 (RGB), 4 (RGBA) |
| **H** | Height in pixels | 224, 256, 512, etc. |

| **W** | Width in pixels | 224, 256, 512, etc. || Normalized | ~[-2, 2] | After ImageNet normalization |

| `torch.float32` | [0.0, 1.0] | Neural network input |

### Batched Images: (B, C, H, W)| `torch.uint8` | [0, 255] | Storage, I/O |

|-------|-------|----------|

When training, images are batched:| dtype | Range | Use Case |

- **B** = Batch size (32, 64, 128, etc.)

- Example: `(32, 3, 224, 224)` = 32 RGB images of size 224×224### Data Types and Ranges:



### Why Channels First?```

plt.imshow(image.permute(1, 2, 0))  # CORRECT - swap to (H, W, C)

PyTorch uses "channels first" format for performance:plt.imshow(image)  # WRONG if image is (C, H, W)

- GPU convolutions are faster with this layout# Matplotlib expects (H, W, C)

- Consecutive channel values are in memory together```python


### Common Gotcha:

In [ ]:
# Create a synthetic RGB image (channels first: C, H, W)
# Red channel
red = torch.zeros(64, 64)
red[:32, :] = 255  # Top half red

# Green channel
green = torch.zeros(64, 64)
green[:, 32:] = 255  # Right half green

# Blue channel
blue = torch.zeros(64, 64)
blue[32:, :32] = 255  # Bottom-left quarter blue

# Stack to create RGB image: (3, 64, 64)
synthetic_image = torch.stack([red, green, blue]).to(torch.uint8)

print(f"Image shape: {synthetic_image.shape}")
print(f"Image dtype: {synthetic_image.dtype}")
print(f"Value range: [{synthetic_image.min()}, {synthetic_image.max()}]")

# Display - need to permute to (H, W, C) for matplotlib
plt.figure(figsize=(6, 6))
plt.imshow(synthetic_image.permute(1, 2, 0).numpy())
plt.title("Synthetic RGB Image (64x64)")
plt.axis("off")
plt.show()

In [ ]:
# Understand image dtypes and value ranges
print("=== Image Data Types ===")
print()
print("torch.uint8:  Values in [0, 255]   - Standard for storage")
print("torch.float32: Values in [0.0, 1.0] - Standard for neural networks")
print()

# Convert between formats
img_uint8 = synthetic_image.clone()  # [0, 255] uint8
img_float = img_uint8.float() / 255.0  # [0, 1] float32

print(
    f"uint8 image: dtype={img_uint8.dtype}, range=[{img_uint8.min()}, {img_uint8.max()}]"
)
print(
    f"float image: dtype={img_float.dtype}, range=[{img_float.min():.2f}, {img_float.max():.2f}]"
)

## 2. Loading Images

torchvision provides multiple ways to load images:
- `torchvision.io.decode_image()` - Fast C++ decoder
- PIL Image + transforms - Traditional approach
- Built-in datasets - MNIST, CIFAR, ImageNet, etc.

In [ ]:
# Create a sample image to work with
sample_img_path = Path("/tmp/sample_image.jpg")

# Create a gradient image
h, w = 256, 256
x = torch.linspace(0, 1, w).unsqueeze(0).expand(h, -1)
y = torch.linspace(0, 1, h).unsqueeze(1).expand(-1, w)

# Create RGB channels with gradients
r = (x * 255).to(torch.uint8)
g = (y * 255).to(torch.uint8)
b = ((1 - x) * 255).to(torch.uint8)

gradient_img = torch.stack([r, g, b])  # (3, H, W)

# Save using PIL
pil_img = Image.fromarray(gradient_img.permute(1, 2, 0).numpy())
pil_img.save(sample_img_path)

print(f"Saved sample image to {sample_img_path}")

# Load using different methods
print("\n=== Loading Methods ===")

# Method 1: torchvision.io (recommended for performance)
img_tensor = decode_image(str(sample_img_path))
print(f"decode_image: shape={img_tensor.shape}, dtype={img_tensor.dtype}")

# Method 2: PIL + convert
pil_loaded = Image.open(sample_img_path)
img_from_pil = torch.from_numpy(np.array(pil_loaded)).permute(2, 0, 1)
print(f"PIL+convert:  shape={img_from_pil.shape}, dtype={img_from_pil.dtype}")

In [ ]:
# Load built-in dataset (MNIST - will download if not present)
mnist_train = datasets.MNIST(
    root="./datasets",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

print(f"MNIST dataset size: {len(mnist_train)}")

# Get a sample
img, label = mnist_train[0]
print(f"Sample image shape: {img.shape}")
print(f"Sample label: {label}")

# Display some samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img, label = mnist_train[i]
    ax.imshow(img.squeeze().numpy(), cmap="gray")
    ax.set_title(f"Label: {label}")
    ax.axis("off")
plt.suptitle("MNIST Sample Images")
plt.tight_layout()
plt.show()

## 3. Modern Transforms (v2 API)

torchvision.transforms.v2 is the recommended API as of 2024:
- Faster performance
- Supports images, bounding boxes, masks
- Better for detection and segmentation tasks

### Key Transforms Categories:
- **Geometry**: Resize, Crop, Rotate, Flip
- **Color**: ColorJitter, Grayscale, Normalize
- **Composition**: Compose, RandomApply, RandomChoice

In [ ]:
# Create a sample image for transform demos
# Use CIFAR-10 for more interesting images
cifar_train = datasets.CIFAR10(root="./datasets", train=True, download=True)

# Get a sample image as tensor
pil_sample, label = cifar_train[0]
sample_tensor = v2.functional.to_image(pil_sample)

print(f"CIFAR-10 classes: {cifar_train.classes}")
print(f"Sample class: {cifar_train.classes[label]}")
print(f"Image shape: {sample_tensor.shape}")

In [ ]:
# Demonstrate geometric transforms
geometric_transforms = {
    "Original": None,
    "Resize(64)": v2.Resize(64),
    "CenterCrop(24)": v2.CenterCrop(24),
    "RandomCrop(24)": v2.RandomCrop(24),
    "HorizontalFlip": v2.RandomHorizontalFlip(p=1.0),
    "Rotate(45°)": v2.RandomRotation(degrees=(45, 45)),
}

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, (name, transform) in zip(axes.flat, geometric_transforms.items()):
    if transform is None:
        result = sample_tensor
    else:
        result = transform(sample_tensor)

    ax.imshow(result.permute(1, 2, 0).numpy())
    ax.set_title(f"{name}\nShape: {tuple(result.shape)}")
    ax.axis("off")

plt.suptitle("Geometric Transforms", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate color transforms
color_transforms = {
    "Original": None,
    "Grayscale": v2.Grayscale(num_output_channels=3),
    "ColorJitter": v2.ColorJitter(
        brightness=0.5, contrast=0.5, saturation=0.5, hue=0.3
    ),
    "GaussianBlur": v2.GaussianBlur(kernel_size=5),
    "RandomInvert": v2.RandomInvert(p=1.0),
    "RandomPosterize": v2.RandomPosterize(bits=2, p=1.0),
}

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, (name, transform) in zip(axes.flat, color_transforms.items()):
    if transform is None:
        result = sample_tensor
    else:
        result = transform(sample_tensor)

    # Handle grayscale display
    if result.shape[0] == 1:
        ax.imshow(result.squeeze().numpy(), cmap="gray")
    else:
        ax.imshow(result.permute(1, 2, 0).numpy())
    ax.set_title(name)
    ax.axis("off")

plt.suptitle("Color Transforms", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Standard ImageNet normalization values
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Create a standard training pipeline
train_transforms = v2.Compose(
    [
        v2.ToImage(),  # Ensure tensor format
        v2.RandomResizedCrop(size=(224, 224), antialias=True),
        v2.RandomHorizontalFlip(p=0.5),
        v2.ColorJitter(brightness=0.2, contrast=0.2),
        v2.ToDtype(torch.float32, scale=True),  # Convert to float [0, 1]
        v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

# Create a standard validation pipeline
val_transforms = v2.Compose(
    [
        v2.ToImage(),
        v2.Resize(256, antialias=True),
        v2.CenterCrop(224),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

print("=== Standard Pipelines ===")
print("\nTraining Pipeline:")
print(train_transforms)
print("\nValidation Pipeline:")
print(val_transforms)

In [ ]:
# Visualize augmentation effects (show same image with different augmentations)
pil_sample, label = cifar_train[42]  # Get a different sample

# Only use augmentation transforms (no normalization for visualization)
aug_transforms = v2.Compose(
    [
        v2.ToImage(),
        v2.RandomResizedCrop(size=(32, 32), scale=(0.5, 1.0), antialias=True),
        v2.RandomHorizontalFlip(p=0.5),
        v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        v2.RandomRotation(degrees=15),
    ]
)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# Show original
axes[0, 0].imshow(pil_sample)
axes[0, 0].set_title("Original")
axes[0, 0].axis("off")

# Show augmented versions
for i, ax in enumerate(axes.flat[1:]):
    augmented = aug_transforms(pil_sample)
    ax.imshow(augmented.permute(1, 2, 0).numpy())
    ax.set_title(f"Augmented {i+1}")
    ax.axis("off")

plt.suptitle(
    f"Data Augmentation Examples - Class: {cifar_train.classes[label]}", fontsize=14
)
plt.tight_layout()
plt.show()

## 4. Auto-Augmentation

Modern approaches use learned augmentation policies:
- **AutoAugment**: Policies learned on ImageNet/CIFAR/SVHN
- **RandAugment**: Simpler with same performance
- **TrivialAugment**: Even simpler, strong baseline

In [ ]:
# Compare auto-augmentation methods
from torchvision.transforms import AutoAugmentPolicy

auto_augments = {
    "Original": None,
    "AutoAugment (CIFAR10)": v2.AutoAugment(policy=AutoAugmentPolicy.CIFAR10),
    "RandAugment": v2.RandAugment(num_ops=2, magnitude=9),
    "TrivialAugment": v2.TrivialAugmentWide(),
}

pil_sample, label = cifar_train[100]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for row in range(2):
    for col, (name, aug) in enumerate(auto_augments.items()):
        ax = axes[row, col]
        if aug is None:
            ax.imshow(pil_sample)
        else:
            # Apply augmentation
            result = aug(pil_sample)
            if isinstance(result, torch.Tensor):
                ax.imshow(result.permute(1, 2, 0).numpy())
            else:
                ax.imshow(result)

        if row == 0:
            ax.set_title(name, fontsize=10)
        ax.axis("off")

plt.suptitle("Auto-Augmentation Methods (2 random samples each)", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Data Loading Pipeline

Efficient data loading is crucial for training. Key components:
- **Dataset**: Provides access to samples
- **DataLoader**: Batches, shuffles, and parallelizes loading

In [ ]:
# Create datasets with transforms
cifar_transforms = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
    ]
)

train_dataset = datasets.CIFAR10(
    root="./datasets", train=True, download=True, transform=cifar_transforms
)

test_dataset = datasets.CIFAR10(
    root="./datasets", train=False, download=True, transform=cifar_transforms
)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Classes: {train_dataset.classes}")

In [ ]:
# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,  # Shuffle for training
    num_workers=0,  # Use 0 for notebooks, increase for scripts
    pin_memory=True,  # Faster GPU transfer
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,  # Don't shuffle for evaluation
    num_workers=0,
    pin_memory=True,
)

# Get a batch
images, labels = next(iter(train_loader))

print(f"Batch shape: {images.shape}")  # (batch, channels, height, width)
print(f"Labels shape: {labels.shape}")
print(f"Value range: [{images.min():.2f}, {images.max():.2f}]")

In [ ]:
# Visualize a batch
def show_batch(images, labels, classes, nrow=8):
    """Display a batch of images in a grid."""
    from torchvision.utils import make_grid

    # Make grid
    grid = make_grid(images[: nrow * 2], nrow=nrow, normalize=True, padding=2)

    plt.figure(figsize=(15, 4))
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.axis("off")

    # Print labels
    label_names = [classes[l] for l in labels[: nrow * 2]]
    print("Labels:", label_names[:nrow])
    print("       ", label_names[nrow:])
    plt.show()


show_batch(images, labels, train_dataset.classes)

## 6. CutMix and MixUp

Advanced augmentations that blend images together:
- **MixUp**: Linear interpolation between images and labels
- **CutMix**: Replace patches of one image with another

In [ ]:
# Get a batch without augmentation
images, labels = next(iter(train_loader))
num_classes = len(train_dataset.classes)

# Create CutMix and MixUp transforms
cutmix = v2.CutMix(num_classes=num_classes)
mixup = v2.MixUp(num_classes=num_classes, alpha=0.2)

# Apply CutMix
cutmix_images, cutmix_labels = cutmix(images, labels)
print(f"CutMix labels shape: {cutmix_labels.shape}")  # Now soft labels

# Apply MixUp
mixup_images, mixup_labels = mixup(images, labels)
print(f"MixUp labels shape: {mixup_labels.shape}")

In [ ]:
# Visualize CutMix and MixUp
fig, axes = plt.subplots(3, 6, figsize=(15, 8))

for i in range(6):
    # Original
    axes[0, i].imshow(images[i].permute(1, 2, 0).numpy())
    axes[0, i].set_title(f"{train_dataset.classes[labels[i]]}", fontsize=9)
    axes[0, i].axis("off")

    # CutMix
    axes[1, i].imshow(cutmix_images[i].permute(1, 2, 0).numpy())
    # Get top 2 classes from soft labels
    top2 = cutmix_labels[i].topk(2)
    title = f"{train_dataset.classes[top2.indices[0]]}:{top2.values[0]:.1f}"
    axes[1, i].set_title(title, fontsize=9)
    axes[1, i].axis("off")

    # MixUp
    axes[2, i].imshow(mixup_images[i].permute(1, 2, 0).numpy().clip(0, 1))
    top2 = mixup_labels[i].topk(2)
    title = f"{train_dataset.classes[top2.indices[0]]}:{top2.values[0]:.1f}"
    axes[2, i].set_title(title, fontsize=9)
    axes[2, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=12)
axes[1, 0].set_ylabel("CutMix", fontsize=12)
axes[2, 0].set_ylabel("MixUp", fontsize=12)

plt.suptitle("CutMix and MixUp Augmentations", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Custom Dataset

For your own data, inherit from `torch.utils.data.Dataset`.

In [ ]:
from torch.utils.data import Dataset


class SyntheticImageDataset(Dataset):
    """Example custom dataset generating synthetic images."""

    def __init__(self, num_samples=1000, image_size=32, num_classes=5, transform=None):
        self.num_samples = num_samples
        self.image_size = image_size
        self.num_classes = num_classes
        self.transform = transform

        # Generate random labels
        self.labels = torch.randint(0, num_classes, (num_samples,))

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Generate a synthetic image based on label
        label = self.labels[idx].item()

        # Create different patterns for different classes
        torch.manual_seed(idx)  # Reproducible

        if label == 0:  # Horizontal stripes
            pattern = (
                torch.arange(self.image_size).unsqueeze(1).expand(-1, self.image_size)
            )
            image = (pattern % 8 < 4).float()
        elif label == 1:  # Vertical stripes
            pattern = (
                torch.arange(self.image_size).unsqueeze(0).expand(self.image_size, -1)
            )
            image = (pattern % 8 < 4).float()
        elif label == 2:  # Diagonal
            x = torch.arange(self.image_size).unsqueeze(0).expand(self.image_size, -1)
            y = torch.arange(self.image_size).unsqueeze(1).expand(-1, self.image_size)
            image = ((x + y) % 8 < 4).float()
        elif label == 3:  # Circle
            x = torch.arange(self.image_size).float() - self.image_size / 2
            y = torch.arange(self.image_size).float() - self.image_size / 2
            xx, yy = torch.meshgrid(x, y, indexing="ij")
            image = (xx**2 + yy**2 < (self.image_size / 3) ** 2).float()
        else:  # Random noise
            image = torch.rand(self.image_size, self.image_size)

        # Add some noise
        image = image + 0.1 * torch.randn_like(image)
        image = image.clamp(0, 1)

        # Convert to 3-channel
        image = image.unsqueeze(0).expand(3, -1, -1)

        if self.transform:
            image = self.transform(image)

        return image, label


# Create and test custom dataset
synthetic_dataset = SyntheticImageDataset(num_samples=100)

# Visualize samples from each class
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
class_samples = {i: None for i in range(5)}

for i in range(len(synthetic_dataset)):
    img, label = synthetic_dataset[i]
    if class_samples[label] is None:
        class_samples[label] = img
    if all(v is not None for v in class_samples.values()):
        break

class_names = ["H-Stripes", "V-Stripes", "Diagonal", "Circle", "Noise"]
for label, (ax, name) in enumerate(zip(axes, class_names)):
    ax.imshow(class_samples[label].permute(1, 2, 0).numpy())
    ax.set_title(name)
    ax.axis("off")

plt.suptitle("Custom Synthetic Dataset Examples")
plt.tight_layout()
plt.show()

## Summary

### Key Concepts

| Concept | Description |
|---------|-------------|
| Image Tensor | Shape (C, H, W), dtype uint8 [0,255] or float32 [0,1] |
| transforms.v2 | Modern, fast API for image transforms |
| Augmentation | Random transforms to increase training data variety |
| Normalization | Scale to zero mean, unit variance for training |
| DataLoader | Efficient batching, shuffling, parallel loading |

### Best Practices

1. **Use transforms.v2** - Faster and more flexible than v1
2. **Keep images as uint8** until just before the model
3. **Use antialias=True** for resizing operations
4. **Apply Normalize last** in the transform pipeline
5. **Use RandAugment** or **TrivialAugment** for easy strong augmentation
6. **Consider CutMix/MixUp** for improved regularization

In [ ]:
# Clean up
import shutil

if sample_img_path.exists():
    sample_img_path.unlink()
print("Notebook completed successfully!")